In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
q3_path = os.path.join(path, 'Q3_data.csv')# نباها تدمج الباث تخليه باث واحد
df = pd.read_csv(q3_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
missing_values = df.isnull().sum()
print("Columns with missing values:")
print(missing_values[missing_values > 0])

In [ ]:
# Task 1: Write your code here:
df = df.fillna(df.mean())

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)


In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, "Target")

In [ ]:
print("It is imbalanced")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1)
y = df["Target"]

In [ ]:
pip install catboost

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0 )
f1_scores = []
for train_index, test_index in kf.split(X, y):
   X_train, X_test = X.iloc[train_index], X.iloc[test_index]
   y_train, y_test = y.iloc[train_index], y.iloc[test_index]
# Fit the model
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Calculate F1-score
score = f1_score(y_test, y_pred)
f1_scores.append(score)
print("F1-score for each fold:", f1_scores)
print("Average F1-score:", np.mean(f1_scores))


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import pandas as pd

# Get feature importance from the trained model
feature_importances = model.get_feature_importance()
feature_names = X.columns

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
})

# Sort by importance descending
importance_df = importance_df.sort_values(by='Importance', ascending=False)
plt.figure(figsize=(12,6))
plt.barh(importance_df['Feature'][:20][::-1], importance_df['Importance'][:20][::-1])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 20 Feature Importances")
plt.show()


In [ ]:
# Task 2: Write your code here:

golden_feature = importance_df['Feature'].iloc[0]
print("The most important feature (Golden Feature) is:", golden_feature)


In [ ]:
# Task Bonus: Write your code here:
X_golden = X[[golden_feature]]
model_single = CatBoostClassifier( iterations=500, learning_rate=0.1, depth=6, verbose=0 )
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
from sklearn.metrics import accuracy_score
import numpy as np

# List to store accuracy scores
accuracy_scores = []

# KFold loop
for train_index, test_index in kf.split(X_golden, y):
    X_train, X_test = X_golden.iloc[train_index], X_golden.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train model
    model_single.fit(X_train, y_train)

    # Predict
    y_pred = model_single.predict(X_test)

    # Calculate accuracy
    acc = accuracy_score(y_test, y_pred)
    accuracy_scores.append(acc)

# Print results
print("Accuracy for each fold using only the golden feature:", accuracy_scores)
print("Average accuracy using only the golden feature:", np.mean(accuracy_scores))
print("Average F1-score of full model:", np.mean(f1_scores))
print("Average accuracy using only golden feature:", np.mean(accuracy_scores))


